In [5]:
#compute class weights for the focal loss to focus on rarer pieces
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import json
import pandas as pd
import numpy as np
from glob import glob
from sklearn.utils.class_weight import compute_class_weight

In [6]:
def parse_labels(example):
    feature_description = {
            'labels': tf.io.VarLenFeature(tf.int64),
            'mel_spectrogram': tf.io.FixedLenFeature([], tf.string),
            'song_name': tf.io.FixedLenFeature([], tf.string),
            'segment_idx': tf.io.FixedLenFeature([], tf.int64),
            'total_segments': tf.io.FixedLenFeature([], tf.int64)
        }
    parsed = tf.io.parse_single_example(example, feature_description)
    labels = tf.sparse.to_dense(parsed['labels'])
    return labels

In [7]:
def calculate_class_weights_from_tfrecords(tfrecord_files, num_classes=8, positive_weight_multiplier=2.0):       
    
    # Create dataset to extract only unique song labels
    # use a dictionary to track unique songs and their labels
    unique_song_labels = {}
    
    for file_path in tfrecord_files:
        dataset = tf.data.TFRecordDataset(file_path)
        
        # Define a function to extract song name and labels
        def extract_song_and_labels(example):
            feature_description = {
                'labels': tf.io.VarLenFeature(tf.int64),
                'mel_spectrogram': tf.io.FixedLenFeature([], tf.string),
                'song_name': tf.io.FixedLenFeature([], tf.string),
                'segment_idx': tf.io.FixedLenFeature([], tf.int64),
                'total_segments': tf.io.FixedLenFeature([], tf.int64)
            }
            parsed = tf.io.parse_single_example(example, feature_description)
            song_name = parsed['song_name']
            labels = tf.sparse.to_dense(parsed['labels'])
            return song_name, labels
        
        # Map and collect song names and labels
        song_labels_dataset = dataset.map(extract_song_and_labels)
        
        # Collect unique song labels
        for song_name, labels in song_labels_dataset:
            song_name_str = song_name.numpy().decode('utf-8')
            # Skip augmented songs if needed
            if song_name_str.endswith('_aug'):
                continue
            unique_song_labels[song_name_str] = labels.numpy()
    
    print(f"Found {len(unique_song_labels)} unique songs")
    
    # Convert to numpy array for processing
    all_labels = np.array(list(unique_song_labels.values()))
    
    class_weights_per_genre = []
    for i in range(num_classes):
        y_genre = all_labels[:, i]
        weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_genre)
        class_weights_per_genre.append(weights)

    class_weights_per_genre = np.array(class_weights_per_genre, dtype=np.float32)
    
    # Apply multiplier to positive class weights to emphasize minority class
    class_weights_per_genre[:, 1] *= positive_weight_multiplier
    class_weights_per_genre[5] *= (positive_weight_multiplier) #helping pop
    class_weights_per_genre[6] *= (positive_weight_multiplier) #helping enallaktiko

    #normalize the weights with their mean
    mean = np.mean(class_weights_per_genre)
    class_weights_per_genre = class_weights_per_genre / mean
    
    # Print distribution for verification
    genre_order = ['LAIKO', 'REMPETIKO', 'ENTEXNO', 'ROCK', 'Mod LAIKO', 'POP', 'ENALLAKTIKO', 'HIPHOP/RNB']
    yes_counts = (all_labels == 1).sum(axis=0)
    no_counts = (all_labels == 0).sum(axis=0)
    
    print("Label distribution:")
    for i, genre in enumerate(genre_order):
        print(f"{genre}: {yes_counts[i]} positive, {no_counts[i]} negative")

        print(f"    - Weights: Negative={class_weights_per_genre[i][0]:.3f}, Positive={class_weights_per_genre[i][1]:.3f}")
    
    return class_weights_per_genre

In [8]:
train_files = tf.io.gfile.glob(
    "/home/georgios/Music Analysis/creating_spectrogram_batches/tfrecord_dataset/train/*.tfrecord")
class_weights = calculate_class_weights_from_tfrecords(train_files)
np.save('npy/class_weights_file', class_weights)

Found 873 unique songs
Label distribution:
LAIKO: 352 positive, 521 negative
    - Weights: Negative=0.178, Positive=0.526
REMPETIKO: 112 positive, 761 negative
    - Weights: Negative=0.122, Positive=1.652
ENTEXNO: 276 positive, 597 negative
    - Weights: Negative=0.155, Positive=0.670
ROCK: 160 positive, 713 negative
    - Weights: Negative=0.130, Positive=1.157
Mod LAIKO: 281 positive, 592 negative
    - Weights: Negative=0.156, Positive=0.659
POP: 114 positive, 759 negative
    - Weights: Negative=0.244, Positive=3.247
ENALLAKTIKO: 176 positive, 697 negative
    - Weights: Negative=0.266, Positive=2.103
HIPHOP/RNB: 40 positive, 833 negative
    - Weights: Negative=0.111, Positive=4.626
